# Optimize training with hyperparameter tuning (sweep job)

**Prince Hands-on Lab – Optimize model training in Azure Machine Learning**

The regularization rate (`--reg_rate`) changes how well the model fits. Instead of trying values one at a time by hand, a **sweep job** trains several models, each with a different value, compares them on a metric, and reports the best one.

To finish faster, the trials run **in parallel on the two nodes** of `aml-cluster`. This is how you spread training work over several compute resources.

## Before you start

You need the **azure-ai-ml** package. The **Python 3.10 - SDK v2** kernel already includes it; the cell below makes sure it is installed and up to date.

> If the output ends with *"you may need to restart the kernel"*, click **Restart kernel** in the toolbar (the ⟳ icon), then continue from the **next** cell.

In [ ]:
%pip install --quiet --upgrade azure-ai-ml azure-identity

## Connect to your workspace

On a compute instance, the workspace details (subscription, resource group, workspace name) are already configured, so `from_config()` finds them automatically.

> If a banner asks you to **Authenticate**, click it and sign in, then run the cell again.

In [ ]:
from lab_helpers import get_ml_client, get_sklearn_environment

ml_client = get_ml_client()

## Choose the training environment

An *environment* is the set of packages installed on the cluster before your script runs. This helper picks a current curated scikit-learn environment in your workspace (or builds a small custom one if none exists), so the job doesn't fail because an old environment was retired.

In [ ]:
sklearn_env = get_sklearn_environment(ml_client)

## Define the base command job

A sweep reuses a normal command job as its template. Test the command job first (previous notebook) so you know the script works.

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

job = command(
    code="../src",
    command="python train-model-parameters.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(type=AssetTypes.URI_FILE, path="../data/diabetes-data/diabetes.csv"),
        "reg_rate": 0.01,
    },
    environment=sklearn_env,
    compute="aml-cluster",
    display_name="diabetes-train-sweep-base",
    experiment_name="sweep-diabetes",
)

## Define the search space

Train four models, each with a different regularization rate, using a `Choice` hyperparameter.

In [ ]:
from azure.ai.ml.sweep import Choice

command_job_for_sweep = job(
    reg_rate=Choice(values=[0.01, 0.1, 1, 10]),
)

## Configure and submit the sweep job

- `sampling_algorithm`: `grid` tries every value. (`random` and `bayesian` are also available.)
- `primary_metric`: the metric used to pick the best model. It **must have exactly the same name** the script logs with `mlflow.log_metric()`, which here is **`Accuracy`**.
- `goal`: `Maximize` (higher accuracy is better).
- `set_limits`: 4 trials in total, **2 at the same time** (one per cluster node), stop after 1 hour.

In [ ]:
sweep_job = command_job_for_sweep.sweep(
    compute="aml-cluster",
    sampling_algorithm="grid",
    primary_metric="Accuracy",
    goal="Maximize",
)
sweep_job.experiment_name = "sweep-diabetes"
sweep_job.display_name = "diabetes-sweep-reg-rate"
sweep_job.set_limits(max_total_trials=4, max_concurrent_trials=2, timeout=3600)

returned_sweep_job = ml_client.create_or_update(sweep_job)
print("Sweep job name:", returned_sweep_job.name)
print("Monitor your job at", returned_sweep_job.studio_url)

## Review the results

The sweep takes about **10–20 minutes**. When it's **Completed**, open it in the studio:

- **Overview**: the best trial and its `Accuracy`
- **Trials** tab: all four models side by side. Tick them and use the chart to compare `Accuracy` for each `reg_rate`.
- Click a trial to see its parameters, metrics and ROC curve.

(Optional) Run the next cell after the sweep has completed to print the best trial from here.

In [ ]:
best = ml_client.jobs.get(returned_sweep_job.name)
print("Status:", best.status)
props = best.properties or {}
for key in ("best_child_run_id", "score", "best_metric_status"):
    if key in props:
        print(f"{key}: {props[key]}")
print("Open the Trials tab for full details:", best.studio_url)